# Adım 5 — Özellik Mühendisliği (Feature Engineering)

**Girdi:** Delta Lake Silver katmanı (`/delta/silver/steam_reviews`)

**Çıktı:** Delta Lake Gold katmanı (`/delta/gold/features_table`) — ML modellerine doğrudan beslenebilir formatta

## Üretilen Özellikler (PDF Adım 5: en az 5 özellik)

| # | Özellik | Tip | İş Mantığı |
|---|---------|-----|------------|
| 1 | **TF-IDF vektörü** | Sparse vector (5000 boyut) | Yorum metninin sayısal temsili. Tokenize → stopword temizliği → HashingTF → IDF. Yorumun *konusunu* yakalayan birincil sinyal. |
| 2 | **text_length** | Numeric | Karakter sayısı. Negatif yorumlar tipik olarak daha uzundur (kullanıcılar şikayetlerini detaylı yazar). |
| 3 | **word_count** | Numeric | Kelime sayısı. text_length ile birlikte yorumun *bilgi yoğunluğu* için ek sinyal. |
| 4 | **review_votes** | Numeric | Topluluk onayı. Yüksek oy alan yorumlar genelde kalitelidir ve sentiment'leri daha güvenilirdir. |
| 5 | **uppercase_ratio** | Numeric | BÜYÜK HARF oranı. Yüksek oran genelde öfke / heyecan göstergesidir → negatif yorumlarda daha sık. |
| 6 | **exclamation_count** | Numeric | Ünlem sayısı. Duygusal yoğunluk göstergesi. |

Ayrıca sınıf dengesizliği için **classWeight** kolonu üretilir.

## BÖLÜM 1 — Spark Session ve Silver Katmanı Yükleme

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = (
    SparkSession.builder
    .appName("SteamReviews_FeatureEngineering")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
# Silver katmanını yükle
SILVER_PATH = "/delta/silver/steam_reviews"
silver_df = spark.read.format("delta").load(SILVER_PATH)

print(f"Silver kayıt sayısı: {silver_df.count():,}")
print("\nSilver schema:")
silver_df.printSchema()
silver_df.show(3, truncate=60)

## BÖLÜM 2 — Sayısal Özellik Üretimi

Metin tabanlı sayısal özellikler doğrudan `review_text` üzerinden hesaplanır.

In [ ]:
# UDF olmadan, native Spark SQL fonksiyonları ile hesaplıyoruz (daha hızlı)
df_numeric = (
    silver_df
    # Özellik 2: text_length (karakter sayısı)
    .withColumn("text_length", F.length(F.col("review_text")).cast(DoubleType()))
    # Özellik 3: word_count (boşluk + 1)
    .withColumn(
        "word_count",
        (F.size(F.split(F.col("review_text"), r"\s+"))).cast(DoubleType())
    )
    # Özellik 4: review_votes (zaten numerik, double'a cast)
    .withColumn("review_votes_d", F.col("review_votes").cast(DoubleType()))
    # Özellik 5: uppercase_ratio (büyük harf sayısı / toplam karakter)
    .withColumn(
        "upper_chars",
        F.length(F.regexp_replace(F.col("review_text"), r"[^A-Z]", "")).cast(DoubleType())
    )
    .withColumn(
        "uppercase_ratio",
        F.when(F.col("text_length") > 0, F.col("upper_chars") / F.col("text_length"))
         .otherwise(F.lit(0.0))
    )
    .drop("upper_chars")
    # Özellik 6: exclamation_count
    .withColumn(
        "exclamation_count",
        F.length(F.regexp_replace(F.col("review_text"), r"[^!]", "")).cast(DoubleType())
    )
)

print("Sayısal özelliklerin örnek dağılımı:")
df_numeric.select(
    "label", "text_length", "word_count", "review_votes_d",
    "uppercase_ratio", "exclamation_count"
).describe().show()

## BÖLÜM 3 — TF-IDF Vektörü

Tokenizer → StopWordsRemover → HashingTF → IDF

In [ ]:
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF, VectorAssembler
)
from pyspark.ml import Pipeline

# 1) Tokenize
tokenizer = Tokenizer(inputCol="review_text", outputCol="tokens_raw")

# 2) Stop words temizle (İngilizce)
stop_remover = StopWordsRemover(inputCol="tokens_raw", outputCol="tokens")

# 3) Hashing TF (5000 boyut — bellek/performans dengesi için)
hashing_tf = HashingTF(inputCol="tokens", outputCol="tf_features", numFeatures=5000)

# 4) IDF
idf = IDF(inputCol="tf_features", outputCol="tfidf_features", minDocFreq=5)

# 5) Tüm sayısal + TF-IDF özelliklerini tek vektörde birleştir
assembler = VectorAssembler(
    inputCols=[
        "tfidf_features",
        "text_length",
        "word_count",
        "review_votes_d",
        "uppercase_ratio",
        "exclamation_count",
    ],
    outputCol="features",
    handleInvalid="skip",
)

feature_pipeline = Pipeline(stages=[tokenizer, stop_remover, hashing_tf, idf, assembler])

print("Feature pipeline fit ediliyor (IDF için tüm korpusun taranması gerek)...")
feature_model = feature_pipeline.fit(df_numeric)
print("Pipeline fit tamamlandı.")

In [ ]:
# Tüm veriyi transform et
df_features = feature_model.transform(df_numeric)

# Final tablo: gerekli kolonları seç
df_features_final = df_features.select(
    "app_id",
    "app_name",
    "user_id",
    "timestamp",
    "label",
    "text_length",
    "word_count",
    "review_votes_d",
    "uppercase_ratio",
    "exclamation_count",
    "features",
)

print("Features tablosu schema:")
df_features_final.printSchema()
df_features_final.select("label", "text_length", "word_count", "features").show(3, truncate=80)

## BÖLÜM 4 — Sınıf Ağırlığı (Class Weight)

Steam yorumlarında pozitif/negatif oranı dengesizdir (~%80/%20). Modellerin azınlık sınıfını öğrenmesi için her satıra ters orantılı `classWeight` atanır:

$w_c = \frac{N_{total}}{2 \cdot N_c}$

In [ ]:
# Sınıf sayılarını hesapla
label_counts = (
    df_features_final.groupBy("label").count().collect()
)
label_count_map = {row["label"]: row["count"] for row in label_counts}
total = sum(label_count_map.values())
n_classes = len(label_count_map)

weights = {
    lbl: total / (n_classes * cnt)
    for lbl, cnt in label_count_map.items()
}

print("Sınıf dağılımı ve hesaplanan ağırlıklar:")
for lbl, cnt in label_count_map.items():
    print(f"  label={lbl}: count={cnt:,}  ratio={cnt/total*100:.2f}%  weight={weights[lbl]:.4f}")

# classWeight kolonu ekle
weight_expr = F.when(F.col("label") == 1, F.lit(weights.get(1, 1.0))) \
               .otherwise(F.lit(weights.get(0, 1.0)))

df_with_weights = df_features_final.withColumn("classWeight", weight_expr)
df_with_weights.select("label", "classWeight").distinct().show()

## BÖLÜM 5 — Gold Katmanına Yazma

`features_table` Delta Lake formatında kaydedilir; `step6_ml_models.ipynb` bu tablodan okur.

In [ ]:
GOLD_FEATURES_PATH = "/delta/gold/features_table"

(
    df_with_weights
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_FEATURES_PATH)
)

print(f"✅ Gold features tablosu yazıldı: {GOLD_FEATURES_PATH}")

# Doğrulama
verify_df = spark.read.format("delta").load(GOLD_FEATURES_PATH)
print(f"Yazılan kayıt sayısı: {verify_df.count():,}")
verify_df.printSchema()

In [ ]:
# Özet istatistikler
print("="*60)
print("  FEATURE ENGINEERING ÖZETİ")
print("="*60)
print(f"  Toplam kayıt          : {verify_df.count():,}")
print(f"  Sayısal özellik sayısı: 5 (text_length, word_count, review_votes_d,")
print(f"                          uppercase_ratio, exclamation_count)")
print(f"  TF-IDF boyutu         : 5000")
print(f"  Toplam feature boyutu : 5005")
print(f"  Class weight (label=0): {weights.get(0, 1.0):.4f}")
print(f"  Class weight (label=1): {weights.get(1, 1.0):.4f}")
print("="*60)

# spark.stop()  # Diğer notebook'lar bağımlı, kapatma